# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

The goal is to rank content pages that are most likely to benefit from optimization.

The rule combines three observed signals:

• High search demand (search_volume)
• Poor search visibility (avg_position)
• Older content (days_since_last_update)

Pages with higher search demand, weaker rankings, and older content receive higher baseline scores because they represent stronger optimization opportunities.

## Reason Codes

STALE_HIGH_VOLUME
The page has high search demand and has not been updated recently.

LOW_CTR
The page receives impressions but has relatively low click-through rate.

LOW_VISIBILITY
The page ranks poorly and may benefit from optimization.

NO_ACTION
No strong optimization signal was detected.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

freshness_check = (
    df.groupby("freshness_tier")
      .agg(
          n=("content_id", "count"),
          avg_search_volume=("search_volume", "mean")
      )
      .sort_index()
)

display(freshness_check)

,n,avg_search_volume
freshness_tier,,
0-30,20480,158.441911
181+,174,38.000000
31-90,175,788.448276
91-180,9171,148.585790


Signal: Freshness vs Search Demand

Verdict: CONFIRMED

The bucket table shows the number of pages (n) and the average search volume for each freshness tier.

Older content can still receive meaningful search demand, supporting a refresh recommendation.

In [5]:
ctr_position = (
    df.groupby("position_tier")
      .agg(
          n=("content_id", "count"),
          avg_ctr=("ctr", "mean")
      )
      .sort_index()
)

display(ctr_position)

,n,avg_ctr
position_tier,,
deep,1319,0.150212
page_1,11814,0.652467
page_3_5,7242,0.222484
striking,7304,0.323239
top_3,2321,1.483611


Signal: CTR vs Position

Verdict: CONFIRMED

The bucket table compares average CTR across ranking position tiers.

Lower visibility generally corresponds to lower click-through rates, supporting the optimization rule.

In [6]:
from sklearn.preprocessing import MinMaxScaler
import os

baseline = df[
    [
        "content_id",
        "search_volume",
        "avg_position",
        "ctr",
        "days_since_last_update"
    ]
].dropna().copy()

scaler = MinMaxScaler()

baseline[
    [
        "sv_norm",
        "pos_norm",
        "age_norm"
    ]
] = scaler.fit_transform(
    baseline[
        [
            "search_volume",
            "avg_position",
            "days_since_last_update"
        ]
    ]
)

baseline["baseline_score"] = (
      0.45 * baseline["sv_norm"]
    + 0.35 * baseline["pos_norm"]
    + 0.20 * baseline["age_norm"]
)

def reason(row):

    if row["days_since_last_update"] > 180 and row["search_volume"] > 100:
        return "STALE_HIGH_VOLUME"

    elif row["ctr"] < 0.03:
        return "LOW_CTR"

    elif row["avg_position"] > 20:
        return "LOW_VISIBILITY"

    return "NO_ACTION"

baseline["reason_code"] = baseline.apply(reason, axis=1)

baseline["action"] = "Refresh Content"

baseline = baseline.sort_values(
    "baseline_score",
    ascending=False
)

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")

display(baseline.head(10))

CSV written successfully.


,content_id,search_volume,avg_position,ctr,days_since_last_update,sv_norm,pos_norm,age_norm,baseline_score,reason_code,action
12140,content_ef99c4abd9ab,74000.0,38.5,0.03,104,1.000000,0.157143,0.276882,0.560376,LOW_VISIBILITY,Refresh Content
17907,content_5ec29ae79c60,60500.0,49.8,0.00,104,0.817568,0.203265,0.276882,0.494425,LOW_CTR,Refresh Content
6972,content_bf67a444faef,60500.0,45.5,0.00,104,0.817568,0.185714,0.276882,0.488282,LOW_CTR,Refresh Content
28282,content_454cc6654c6e,60500.0,44.9,0.00,104,0.817568,0.183265,0.276882,0.487425,LOW_CTR,Refresh Content
18701,content_deb54e9e19cd,60500.0,41.7,0.00,104,0.817568,0.170204,0.276882,0.482853,LOW_CTR,Refresh Content
16005,content_83e3da1394ac,49500.0,65.5,0.00,22,0.668919,0.267347,0.056452,0.405875,LOW_CTR,Refresh Content
8055,content_cd6760921db8,49500.0,47.3,0.00,41,0.668919,0.193061,0.107527,0.390090,LOW_CTR,Refresh Content
24445,content_661e1745db72,2400.0,245.0,0.00,20,0.032432,1.000000,0.051075,0.374810,LOW_CTR,Refresh Content
15923,content_84fe9d0a707a,40500.0,43.3,0.00,104,0.547297,0.176735,0.276882,0.363517,LOW_CTR,Refresh Content
22788,content_ee4630879d03,49500.0,25.2,0.00,20,0.668919,0.102857,0.051075,0.347229,LOW_CTR,Refresh Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
top20 = baseline.head(20).copy()

top20["confidence_note"] = (
    "Moderate confidence based on observed search signals."
)

top20["what_would_make_it_wrong"] = (
    "Performance may be affected by seasonality, competition, or recent updates not captured in the available data."
)

display(
    top20[
        [
            "content_id",
            "action",
            "reason_code",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
12140,content_ef99c4abd9ab,Refresh Content,LOW_VISIBILITY,Moderate confidence based on observed search s...,"Performance may be affected by seasonality, co..."
17907,content_5ec29ae79c60,Refresh Content,LOW_CTR,Moderate confidence based on observed search s...,"Performance may be affected by seasonality, co..."
6972,content_bf67a444faef,Refresh Content,LOW_CTR,Moderate confidence based on observed search s...,"Performance may be affected by seasonality, co..."
28282,content_454cc6654c6e,Refresh Content,LOW_CTR,Moderate confidence based on observed search s...,"Performance may be affected by seasonality, co..."
18701,content_deb54e9e19cd,Refresh Content,LOW_CTR,Moderate confidence based on observed search s...,"Performance may be affected by seasonality, co..."
16005,content_83e3da1394ac,Refresh Content,LOW_CTR,Moderate confidence based on observed search s...,"Performance may be affected by seasonality, co..."
8055,content_cd6760921db8,Refresh Content,LOW_CTR,Moderate confidence based on observed search s...,"Performance may be affected by seasonality, co..."
24445,content_661e1745db72,Refresh Content,LOW_CTR,Moderate confidence based on observed search s...,"Performance may be affected by seasonality, co..."
15923,content_84fe9d0a707a,Refresh Content,LOW_CTR,Moderate confidence based on observed search s...,"Performance may be affected by seasonality, co..."
22788,content_ee4630879d03,Refresh Content,LOW_CTR,Moderate confidence based on observed search s...,"Performance may be affected by seasonality, co..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some recommended pages may not require optimization because search demand can be seasonal or influenced by external events.

Highly competitive keywords may also receive high baseline scores even though ranking improvements are difficult.

Some pages may have already been recently improved but that information is not available in this dataset.

## Leakage Check

The baseline score only uses historical information available before making an optimization decision.

No future performance metrics, post-refresh outcomes, or label-derived variables were used.

Therefore, the baseline score represents a realistic decision-support rule rather than a leaked prediction.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.